### MNIST Classifier

In this notebook, we will implement a CNN classifier to classify all the 10 digits of the MNIST dataset. The objective of this lesson is twofold:
* To build our first CNN classifier.
* To use cross-entropy as classification loss.

Let's start with the usual imports.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

from time import time
import numpy as np
import keras
from keras import Model
from keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

# Set the seed for reproducibility (python, numpy and the backend)
keras.utils.set_random_seed(1234)

#### Dataset Loading

We have already inspected the MNIST dataset. We are going to load it now since we are going to use it for training the classifier.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Dataset params
num_classes = 10
size = x_train.shape[1]

# Normalization
x_train = x_train/255
x_test = x_test/255

print('Train set:   ', len(y_train), 'samples')
print('Test set:    ', len(y_test), 'samples')
print('Sample dims: ', x_train.shape)

#### One-Hot Encoding

In the previous notebook we saw that regressing the digit as a single number does not work well: it imposes an order and a distance between classes that does not exist (a 4 is not "closer" to a 5 than to a 9). Instead, the network will have one output per class and we encode each label as a **one-hot vector**: all zeros except a one at the position of the true class. This resembles a probability distribution: the true class has probability 1 and all the others 0.

In [ ]:
print('Label:   ', y_train[0])

y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

print('One-hot: ', y_train[0])

#### Building the Classifier

We are going to build the same CNN classifier as before, with two differences: the last layer (the classification head) now has 10 outputs, one per class, and it uses the **softmax** activation.

The outputs of an arbitrary layer are just numbers. Softmax turns them into a proper probability distribution: it exponentiates the outputs and normalizes them by their sum, so all the values are in [0, 1] and add up to 1. That is exactly what we need to compare the network output with a one-hot label.

In [ ]:
inputs = Input(shape=(size, size, 1))

net = Conv2D(16, kernel_size=(3, 3), activation="relu")(inputs)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Conv2D(32, kernel_size=(3, 3), activation="relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Flatten()(net)
outputs = Dense(num_classes, activation="softmax")(net)

model = Model(inputs, outputs)
model.summary()

#### Training

Let's now compile and train the model. We will use the **categorical cross-entropy** as loss. For one sample, it is just the negative logarithm of the probability that the network assigns to the true class:

$$L = -\sum_c y_c \log(p_c) = -\log(p_{true})$$

If the network is sure about the right class ($p_{true} \to 1$) the loss goes to 0. If it assigns a tiny probability to the right class, the loss explodes. So cross-entropy pushes the network to be not only correct but also *confident*. And note that we can use the built-in accuracy metric for monitoring the training.

A side note: one-hot encoding the labels is not strictly necessary. If we keep the labels as integers (the class indices), the `sparse_categorical_crossentropy` loss computes exactly the same cross-entropy and does the encoding internally.

In [ ]:
epochs = 25
batch_size = 128

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

start = time()
history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1, verbose=2)
print('Elapsed time', time() - start)

Let's now plot the history to see the evolution of the training.

In [ ]:
def plot_history(history):
    h = history.history
    epochs = range(len(h['loss']))

    plt.subplot(121), plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
    plt.legend(['Train', 'Validation'])
    plt.subplot(122), plt.plot(epochs, h['accuracy'], '.-',
                               epochs, h['val_accuracy'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Accuracy')
    plt.legend(['Train', 'Validation'])
        
    print('Train Acc     ', h['accuracy'][-1])
    print('Validation Acc', h['val_accuracy'][-1])
    
plot_history(history)

We have obtained an overall accuracy of about 99%. This might seem lower than we obtained using the binary classifier before but remember we are now classifying **all 10 digits**.

Note also that the training loss keeps decreasing while the validation loss stops improving after about 10 epochs, and the training accuracy ends up clearly above the validation one. The network is starting to memorize the training samples. This is called overfitting and we will deal with it in the next lessons.

#### Evaluation

Let's now look at the network predictions. Each prediction is a vector of 10 probabilities, and the predicted class is the one with the highest probability.

To see how softmax and cross-entropy work together, we are going to compute the loss of one test sample by hand and compare it to the Keras implementation. We pick a misclassified sample, since for a correctly (and confidently) classified one the loss is practically zero.

In [ ]:
probs = model.predict(x_test, verbose=0)
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(probs, axis=1)

# First misclassified sample
idx = np.flatnonzero(y_pred != y_true)[0]

print('Probabilities:', probs[idx].round(3))
print('Sum:          ', probs[idx].sum())
print('True', y_true[idx], '| Pred', y_pred[idx])

# Cross-entropy: negative log of the probability of the true class
print('Cross-entropy (by hand):', -np.log(probs[idx][y_true[idx]]))
print('Cross-entropy (Keras):  ', float(keras.losses.categorical_crossentropy(y_test[idx], probs[idx])))

In [ ]:
digits = range(0, 10)

for digit in digits:
    mask = y_true == digit
    
    tp = np.sum(y_pred[mask] == digit)
    total = np.sum(mask)
    
    print('Digit-', digit, ' acc', tp/total)

The trained model can also be quickly evaluated by calling the evaluate method on the test data. It will return the loss and the metric (or metrics).

In [ ]:
ev = model.evaluate(x_test, y_test, verbose=0)
print('Test loss  ', ev[0])
print('Test metric', ev[1])

Random test samples are almost always classified correctly, so they are not very informative. Let's rather visualise some of the errors. In the titles, we show the probability that the network assigns to its (wrong) prediction.

In [ ]:
errors = np.flatnonzero(y_pred != y_true)
print(len(errors), 'misclassified samples out of', len(y_true))

for ii, idx in enumerate(np.random.choice(errors, 15, replace=False)):
    plt.subplot(3,5,ii+1), plt.imshow(x_test[idx, ...], cmap='gray')
    plt.title('True: ' + str(y_true[idx]) + ' | Pred: ' + str(y_pred[idx]) + ' (%.2f)' % probs[idx, y_pred[idx]])

Note that many of these digits are really ambiguous, even for a human, and that the network is often very confident about its wrong prediction (probabilities close to 1). This is a side effect of training with cross-entropy, which pushes the network towards confident outputs.